In [1]:
!pip install pytorch-lightning monai torchmetrics scikit-image
import os
import numpy as np
import pandas as pd
import glob
from PIL import Image
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as T
import pytorch_lightning as pl
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import InterpolationMode
from monai.transforms import Activations
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.metrics import f1_score
from torchmetrics import F1Score
from torch.optim.lr_scheduler import StepLR
from pytorch_lightning.callbacks import ModelCheckpoint
import scipy.ndimage as ndi
import cv2
from tqdm import tqdm
from skimage.filters import threshold_otsu
from scipy.stats import kurtosis, skew

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 68.5 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.8.93
    Uninstalling nvidia-nvjitlink-cu12-12.8.93:
      Successfully uninstalled nvidia-nvjitlink-cu12-12.8.93
  Attempting uninstall: nvidia-curand-cu12
    Found existing installation: nvidia-curand-cu12 10.3.9.90
    Uninstalling nvidia-curand-cu12-10.3.

2025-04-29 14:20:17.175063: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1745936417.377402      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745936417.435710      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [2]:
# === Transformations ===
train_transforms = T.Compose([
    T.Resize(224, interpolation=InterpolationMode.BICUBIC),
    T.RandomResizedCrop(224),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.RandomRotation(20),
    T.GaussianBlur(kernel_size=(7, 13), sigma=(0.1, 1.0)),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = T.Compose([
    T.Resize(224, interpolation=InterpolationMode.BICUBIC),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


In [3]:
# === Custom Dataset Classes ===
class ImageDataset(Dataset):
    def __init__(self, file_list, labels=None, transform=None):
        self.file_list = file_list
        self.labels = labels
        self.transform = transform

    def __len__(self): return len(self.file_list)

    def __getitem__(self, idx):
        img = Image.open(self.file_list[idx]).convert("RGB")
        img = self.transform(img) if self.transform else img
        return (img, self.labels[idx]) if self.labels is not None else img

class TestImageDataset(Dataset):
    def __init__(self, file_list, transform=None):
        self.file_list = file_list
        self.transform = transform

    def __len__(self): return len(self.file_list)

    def __getitem__(self, idx):
        img_path = self.file_list[idx]
        img = Image.open(img_path).convert("RGB")
        return self.transform(img), os.path.basename(img_path)

In [4]:
# === LightningModule Wrapper ===
class Net(pl.LightningModule):
    def __init__(self, model, optimizer, scheduler, train_loader, val_loader):
        super().__init__()
        self.model = model
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.loss_fn = nn.BCEWithLogitsLoss()
        self.metric = F1Score(task='binary')
        self.activation = Activations(sigmoid=True)

    def forward(self, x): return self.model(x)
    def train_dataloader(self): return self.train_loader
    def val_dataloader(self): return self.val_loader
    def configure_optimizers(self):
        return {'optimizer': self.optimizer, 'lr_scheduler': self.scheduler}

    def training_step(self, batch, batch_idx):
        x, y = batch
        y = y[:, None].float()
        logits = self(x)
        loss = self.loss_fn(logits, y)
        self.log("train_loss", loss)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        y = y[:, None].float()
        logits = self(x)
        loss = self.loss_fn(logits, y)
        f1 = self.metric(self.activation(logits), y)
        self.log_dict({"val_loss": loss, "f1": f1})

In [5]:
# === Dataset Loading + Train/Test Setup ===
def load_test_df():
    df = pd.read_csv('/kaggle/input/ai-vs-human-generated-dataset/test.csv')
    df['id'] = df['id'].apply(lambda x: f"/kaggle/input/ai-vs-human-generated-dataset/{x}")
    return df

def get_train_valid_split():
    df = pd.read_csv('/kaggle/input/ai-vs-human-generated-dataset/train.csv')
    df['file_name'] = df['file_name'].apply(lambda x: f"/kaggle/input/ai-vs-human-generated-dataset/{x}")
    return train_test_split(df['file_name'].values, df['label'].values, test_size=0.05, random_state=43, shuffle=False)

def create_dataloaders(train_paths, val_paths, train_labels, val_labels):
    bs = 32
    return (
        DataLoader(ImageDataset(train_paths, train_labels, train_transforms), batch_size=bs, shuffle=True, num_workers=4),
        DataLoader(ImageDataset(val_paths, val_labels, val_transforms), batch_size=bs, shuffle=False, num_workers=4)
    )

def create_model_optimizer_scheduler():
    model = models.convnext_base(weights="DEFAULT")
    for p in model.features.parameters(): p.requires_grad = False
    for p in model.features[-2:].parameters(): p.requires_grad = True

    model.classifier = nn.Sequential(
        nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten(),
        nn.BatchNorm1d(1024), nn.Linear(1024, 512), nn.ReLU(), nn.Dropout(0.4), nn.Linear(512, 1)
    )

    optimizer = torch.optim.AdamW([
        {'params': model.features[-2:].parameters(), 'lr': 1e-5},
        {'params': model.classifier.parameters(), 'lr': 1e-4}
    ])
    return model.cuda(), optimizer, StepLR(optimizer, step_size=5, gamma=0.7)


In [6]:
# === Train Model ===
def train_model(model, optimizer, scheduler, train_loader, val_loader):
    net = Net(model, optimizer, scheduler, train_loader, val_loader)
    trainer = pl.Trainer(
        devices=1, max_epochs=10, log_every_n_steps=16,
        callbacks=[ModelCheckpoint(dirpath='models/', monitor='f1', mode='max')]
    )
    trainer.fit(net)
    return net

In [7]:
# === Predict ===
def predict(model, df_test):
    model.eval().cuda()
    ds = TestImageDataset(df_test['id'].values, transform=val_transforms)
    loader = DataLoader(ds, batch_size=32, shuffle=False, num_workers=4)
    preds, names = [], []
    with torch.no_grad():
        for x, name in tqdm(loader):
            logits = model(x.cuda())
            preds.extend(logits.sigmoid().cpu().numpy().flatten())
            names.extend([f"test_data_v2/{n}" for n in name])
    return pd.DataFrame({'id': names, 'label': (np.array(preds) > 0.5).astype(int), 'logits': preds})


In [8]:
# === Run All Steps ===
train_paths, val_paths, train_labels, val_labels = get_train_valid_split()
train_loader, val_loader = create_dataloaders(train_paths, val_paths, train_labels, val_labels)
model, optimizer, scheduler = create_model_optimizer_scheduler()
net = train_model(model, optimizer, scheduler, train_loader, val_loader)
df_test = load_test_df()
cnn_prediction_df = predict(net, df_test)

Downloading: "https://download.pytorch.org/models/convnext_base-6075fbad.pth" to /root/.cache/torch/hub/checkpoints/convnext_base-6075fbad.pth
100%|██████████| 338M/338M [00:01<00:00, 211MB/s]


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

100%|██████████| 174/174 [01:37<00:00,  1.79it/s]


In [9]:
# === Feature Extraction (KMeans) ===
def get_stats(file):
    img = cv2.imread(file)
    h, w = img.shape[:2]
    ratio = max(h, w) / min(h, w)
    size = os.stat(file).st_size / 1024**2
    smooth = np.mean(np.abs(ndi.laplace(img.astype(float)/255)))
    noise = np.mean(np.abs(ndi.median_filter(img.astype(float), 3) - img))
    base = [h, w, ratio, size, smooth, noise]
    for i in range(3):
        channel = img[:,:,i]
        stats = [channel.mean(), channel.std(), *np.quantile(channel, [0.25, 0.75]),
                 channel.max(), channel.min(), threshold_otsu(channel), skew(channel.flatten())]
        base.extend(stats)
    return base

files = glob.glob('/kaggle/input/ai-vs-human-generated-dataset/test_data_v2/**')
stats = np.array([get_stats(f) for f in tqdm(files)])
df_kmeans = pd.DataFrame({'id': [f"test_data_v2/{os.path.basename(f)}" for f in files]})
df_kmeans['label'] = KMeans(2, random_state=2).fit_predict(stats)


100%|██████████| 5540/5540 [1:40:44<00:00,  1.09s/it]
/usr/local/lib/python3.11/dist-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(


In [10]:
# Align labels with CNN predictions
if (df_kmeans.sort_values('id')['label'].values == cnn_prediction_df.sort_values('id')['label'].values).mean() < 0.5:
    df_kmeans['label'] = 1 - df_kmeans['label']

In [11]:
# Inject confident CNN predictions
confident = cnn_prediction_df.sort_values('logits')
df_kmeans.loc[df_kmeans['id'].isin(confident.tail(1500)['id']), 'label'] = 1
df_kmeans.loc[df_kmeans['id'].isin(confident.head(50)['id']), 'label'] = 0

# === Save Final Submission ===
df_kmeans[['id', 'label']].to_csv('submission.csv', index=False)
print("✅ Final Ensemble Submission Saved as submission.csv")

✅ Final Ensemble Submission Saved as submission.csv
